# Métodos Numéricos II
## Práctica: El efecto mariposa en las ecuaciones de Lorenz

**Integrantes del grupo:** Miriam García Sollo y Juana María Rascón Contreras  
**Asignatura:** Métodos Numéricos II, Universidad de Granada  

---

### Introducción y objetivos

El objetivo principal de esta práctica es estudiar numéricamente el sistema de Lorenz mediante el método de Runge-Kutta clásico de orden 4. Fue propuesto por el meteorólogo Edward Lorenz en 1963 para modelar la convección atmosférica (Lorenz, 1963): en él, $x$ representa la intensidad de la convección, $y$ la diferencia de temperatura entre corrientes ascendentes y descendentes, y $z$ la desviación del perfil de temperatura respecto a un perfil lineal.

El sistema viene dado por

$$
\begin{cases}
x'(t)=\sigma(y(t)-x(t)),\\
y'(t)=x(t)(\rho-z(t))-y(t),\\
z'(t)=x(t)y(t)-\beta z(t).
\end{cases}
$$

Como vemos en los apuntes de la asignatura, se trata de un sistema **autónomo** porque el campo vectorial $\mathbf{F}$ no depende explícitamente de $t$. Aunque es determinista, para ciertos parámetros muestra comportamiento caótico: dos trayectorias que empiezan casi en el mismo punto se separan de forma exponencial, dando lugar al llamado *efecto mariposa*.

### Tareas a desarrollar

Para alcanzar el objetivo, estructuramos el trabajo en los siguientes apartados:

* **Apartado 1: Resolución con RK4.** Implementación del método de Runge-Kutta clásico de orden 4 y representación de las soluciones aproximadas para los parámetros del enunciado, partiendo de $(x_0,y_0,z_0)=(3,15,1)$.
* **Apartado 2: Sensibilidad a las condiciones iniciales.** Repetición del proceso con distintas condiciones iniciales para ilustrar el efecto mariposa y comentar el resultado.
* **Apartado 3: Variación del parámetro $\rho$.** Comparación del comportamiento para $\rho=14$, $\rho=28$ y $\rho=99.6$, analizando las transiciones entre regímenes estable y caótico.
* **Aportes personales:** Trayectoria coloreada por velocidad del flujo, diagrama de bifurcaciones, estimación del exponente de Lyapunov, verificación del orden de convergencia, selector interactivo de zona y cámara, animación del avance de iteraciones, y mapa local de sensibilidad.

In [ ]:
# Librerias principales: numpy para calculos, matplotlib para graficas
# matplotlib para todo lo relacionado con las gráficas
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import animation
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML
# joblib lo usamos para paralelizar los cálculos más pesados del notebook
from joblib import Parallel, delayed

# ipywidgets para los sliders interactivos, lo intentamos importar
# pero si no está disponible el notebook sigue funcionando igual
try:
    import ipywidgets as widgets
    from ipywidgets import interact
    WIDGETS_DISPONIBLES = True
except Exception:
    WIDGETS_DISPONIBLES = False

# Paleta de colores fija para que todas las figuras sean coherentes entre sí
# Así no tenemos que recordar los códigos hex cada vez que hacemos una gráfica
AZUL    = "#2b4fa8"
ROJO    = "#c0392b"
VERDE   = "#1a7a4a"
NARANJA = "#d4670a"
LILA    = "#6a3fa0"
GRIS    = "#555555"

# Configuración global de matplotlib para que todas las figuras
# salgan con el mismo estilo sin tener que repetirlo en cada plot
plt.rcParams.update({
    "figure.figsize"    : (9, 5),
    "axes.grid"         : True,
    "grid.alpha"        : 0.2,
    "grid.linestyle"    : "--",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "font.size"         : 11,
    "axes.prop_cycle"   : plt.cycler(color=[AZUL, ROJO, VERDE, NARANJA, LILA]),
    "figure.dpi"        : 110,
})

np.set_printoptions(precision=6, suppress=True)
print("Librerías cargadas correctamente.")

---
## 1. Apartado 1: resolución con RK4
### 1.1 Justificación teórica y esquema numérico

El sistema de Lorenz es un problema de valores iniciales (PVI) de la forma

$$
\mathbf{u}'(t)=\mathbf{F}(t,\mathbf{u}(t)), \qquad \mathbf{u}(t_0)=\mathbf{u}_0,
$$

donde $\mathbf{u}(t)=(x(t),y(t),z(t))^\top$. Como $\mathbf{F}$ no depende de $t$, el sistema es **autónomo** tal y como lo definimos en los apuntes. La existencia y unicidad de solución la garantiza el **Teorema de Picard-Lindelöf**: como $\mathbf{F}\in C^\infty(\mathbb{R}^3)$ es localmente Lipschitziana en $\mathbf{u}$, existe exactamente una solución para cada condición inicial. La **no linealidad** del sistema es la responsable del comportamiento caótico: los términos $xz$ en la segunda ecuación y $xy$ en la tercera introducen el acoplamiento que lo hace posible.

Tal y como vimos en teoría, un método de un paso tiene **orden $p$** si el error de discretización local satisface

$$
\mathcal{E}(h)=\frac{1}{h}\bigl(x(t+h)-x(t)-h\Phi(t,x(t),h)\bigr)=O(h^p).
$$

El método RK4 tiene orden 4, es **consistente** y **estable** (la función de incremento $\Phi$ verifica una condición de Lipschitz en $\mathbf{u}$). Por el teorema visto en los apuntes, todo método consistente y estable es **convergente**: la solución numérica converge a la exacta cuando $h\to 0$. Estas propiedades garantizan que la implementación es matemáticamente sólida.

El esquema viene dado por

$$
\mathbf{u}_{n+1}=\mathbf{u}_n+\frac{h}{6}(K_1+2K_2+2K_3+K_4),
$$

con $K_1=\mathbf{F}(t_n,\mathbf{u}_n)$, $K_2=\mathbf{F}\!\left(t_n+\tfrac{h}{2},\mathbf{u}_n+\tfrac{h}{2}K_1\right)$, $K_3=\mathbf{F}\!\left(t_n+\tfrac{h}{2},\mathbf{u}_n+\tfrac{h}{2}K_2\right)$, $K_4=\mathbf{F}(t_n+h,\mathbf{u}_n+hK_3)$.

En términos de la **tabla de Butcher**, que como vimos en teoría organiza de forma compacta los coeficientes de cualquier método Runge-Kutta:

$$
\begin{array}{c|cccc}
0   &     &     &     &   \\
1/2 & 1/2 &     &     &   \\
1/2 &  0  & 1/2 &     &   \\
1   &  0  &  0  &  1  &   \\
\hline
    & 1/6 & 1/3 & 1/3 & 1/6
\end{array}
$$

Hemos implementado el método nosotras mismas, sin recurrir a solvers externos, para aplicar directamente el esquema visto en clase.

### 1.2 Implementación

In [ ]:
# =============================================================================
# NÚCLEO NUMÉRICO: implementamos desde cero el campo de Lorenz y RK4
# =============================================================================

def lorenz(t, u, sigma=10.0, rho=28.0, beta=8/3):
    """
    Campo vectorial del sistema de Lorenz.
    Traduce directamente las tres ecuaciones diferenciales del enunciado.

    Parámetros
    ----------
    t : float
        Variable temporal. El sistema es autónomo (F no depende de t),
        pero lo incluimos por coherencia con la interfaz general de los PVI.
    u : array_like, shape (3,)
        Vector de estado (x, y, z).
    sigma : float
        Numero de Prandtl: cociente entre viscosidad y difusividad termica.
    rho : float
        Numero de Rayleigh reducido: controla si hay conveccion o no.
    beta : float
        Parámetro geométrico del modo convectivo más inestable.
    """
    x, y, z = u
    # Simplemente despejamos cada derivada en las tres ecuaciones del sistema
    return np.array([
        sigma * (y - x),       # dx/dt
        x * (rho - z) - y,     # dy/dt, termino no lineal: x*z
        x * y - beta * z       # dz/dt, termino no lineal: x*y
    ], dtype=float)


def paso_rk4(f, t, u, h, **params):
    """
    Un único paso del método de Runge-Kutta clásico de orden 4.
    Implementamos el esquema del temario: cuatro evaluaciones del campo
    combinadas con pesos (1, 2, 2, 1)/6, parecido a la regla de Simpson.
    """
    # K1: pendiente al inicio del intervalo, con la información que tenemos
    k1 = f(t,       u,           **params)
    # K2: avanzamos hasta el punto medio usando K1 como estimación
    k2 = f(t + h/2, u + h*k1/2, **params)
    # K3: volvemos al punto medio pero con una estimación mejorada (K2)
    k3 = f(t + h/2, u + h*k2/2, **params)
    # K4: llegamos al final del intervalo usando la mejor estimación del medio
    k4 = f(t + h,   u + h*k3,   **params)
    # Combinamos las cuatro pendientes: los extremos pesan la mitad que el medio
    return u + (h/6) * (k1 + 2*k2 + 2*k3 + k4)


def resolver_rk4(f, u0, intervalo=(0.0, 100.0), h=0.01, **params):
    """
    Resuelve un PVI autónomo con RK4 y paso fijo.
    Devuelve el vector de tiempos t y la matriz U donde cada fila i
    contiene la aproximación (x_i, y_i, z_i) en el instante t_i.
    """
    t0, tf = intervalo
    n_pasos = int(round((tf - t0) / h))
    # Usamos linspace en lugar de ir sumando h: sumar h muchas veces acumula
    # pequeños errores de redondeo que con linspace se evitan directamente
    t = np.linspace(t0, tf, n_pasos + 1)
    U = np.empty((n_pasos + 1, len(u0)), dtype=float)
    U[0] = np.array(u0, dtype=float)   # condición inicial en la primera fila
    for n in range(n_pasos):
        U[n + 1] = paso_rk4(f, t[n], U[n], h, **params)
    return t, U


def puntos_equilibrio_lorenz(sigma=10.0, rho=28.0, beta=8/3):
    """
    Calcula los puntos de equilibrio igualando F(u) = 0 y despejando.
    Para rho > 1 hay tres: el origen y los simétricos C+ y C-.
    Son estables si rho < rho_H ≈ 24.74 e inestables si rho > rho_H.
    """
    eq = [np.array([0.0, 0.0, 0.0])]   # el origen siempre es equilibrio
    if rho > 1:
        # Estos valores salen de resolver el sistema F(u) = 0 a mano
        a = np.sqrt(beta * (rho - 1))
        eq.extend([np.array([ a,  a, rho - 1]),
                   np.array([-a, -a, rho - 1])])
    return np.array(eq)


def representar_3d(ax, U, titulo, color=AZUL, alpha=0.85, salto=1):
    """
    Dibuja una trayectoria 3D marcando inicio (negro) y fin (rojo).
    El parámetro salto reduce los puntos dibujados sin alterar el cálculo.
    """
    V = U[::salto]
    ax.plot(V[:, 0], V[:, 1], V[:, 2], lw=0.6, color=color, alpha=alpha)
    ax.scatter(*U[0],  color="#222222", s=30, zorder=5, label="inicio")
    ax.scatter(*U[-1], color=ROJO,      s=30, zorder=5, label="final")
    ax.set_title(titulo, pad=8)
    ax.set_xlabel("x(t)", labelpad=4)
    ax.set_ylabel("y(t)", labelpad=4)
    ax.set_zlabel("z(t)", labelpad=4)
    ax.view_init(elev=24, azim=-56)


def resumen_trayectoria(nombre, t, U):
    """Imprime un pequeño resumen numérico de la trayectoria."""
    mins, maxs, final = U.min(axis=0), U.max(axis=0), U[-1]
    print(f"{nombre}")
    print(f"  punto final : ({final[0]: .5f}, {final[1]: .5f}, {final[2]: .5f})")
    print(f"  rango x     : [{mins[0]: .3f}, {maxs[0]: .3f}]")
    print(f"  rango y     : [{mins[1]: .3f}, {maxs[1]: .3f}]")
    print(f"  rango z     : [{mins[2]: .3f}, {maxs[2]: .3f}]")


print("Núcleo numérico definido correctamente.")

### 1.3 Representación de las soluciones

Usamos $h=0.01$ en el intervalo $[0,100]$, lo que supone $N=10\,000$ pasos de RK4. Representamos tres vistas complementarias:

- Las **series temporales** $x(t)$, $y(t)$, $z(t)$.
- La **trayectoria 3D**, que revela la geometría global del atractor.
- Las **proyecciones** sobre los planos $xy$ y $xz$, coloreadas por tiempo para mostrar en qué instante pasa la trayectoria por cada zona.

In [ ]:
# Parametros del primer apartado, los que pide el enunciado
h               = 0.01
intervalo       = (0.0, 100.0)
u0_base         = np.array([3.0, 15.0, 1.0])
params_caoticos = {"sigma": 10.0, "rho": 28.0, "beta": 8/3}

# Resolvemos: 10 000 pasos de RK4
t, U = resolver_rk4(lorenz, u0_base, intervalo=intervalo, h=h, **params_caoticos)
resumen_trayectoria("Caso base: sigma=10, rho=28, beta=8/3", t, U)

fig = plt.figure(figsize=(16, 11))
gs  = fig.add_gridspec(2, 2, height_ratios=[1, 1.05], hspace=0.35, wspace=0.3)

# Panel 1: series temporales de las tres componentes
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t, U[:, 0], lw=0.75, color=AZUL,    label="x(t)")
ax1.plot(t, U[:, 1], lw=0.75, color=ROJO,    label="y(t)")
ax1.plot(t, U[:, 2], lw=0.75, color=VERDE,   label="z(t)")
ax1.set_title("Evolución temporal de las tres componentes")
ax1.set_xlabel("t")
ax1.set_ylabel("valor aproximado")
ax1.legend(loc="upper right", ncol=3, framealpha=0.8)

# Panel 2: trayectoria 3D
ax2 = fig.add_subplot(gs[0, 1], projection="3d")
representar_3d(ax2, U, "Trayectoria 3D: atractor de Lorenz", color=AZUL, salto=2)
ax2.legend(fontsize=8)

# Paneles 3 y 4: proyecciones sobre xy y xz coloreadas por tiempo
# La idea es asignar un color diferente a cada segmento según en qué instante
# pasa la trayectoria por ese punto del plano, así se ve la evolución temporal
ax3 = fig.add_subplot(gs[1, 0])
puntos = np.array([U[:, 0], U[:, 1]]).T.reshape(-1, 1, 2)
segs   = np.concatenate([puntos[:-1], puntos[1:]], axis=1)
lc3 = LineCollection(segs, cmap="Blues", norm=mcolors.Normalize(t.min(), t.max()),
                     linewidth=0.4, alpha=0.9)
lc3.set_array(t[:-1])
ax3.add_collection(lc3)
ax3.set_xlim(U[:, 0].min() - 1, U[:, 0].max() + 1)
ax3.set_ylim(U[:, 1].min() - 1, U[:, 1].max() + 1)
fig.colorbar(lc3, ax=ax3, label="t", fraction=0.046, pad=0.04)
ax3.set_title("Proyección sobre el plano xy  (color = tiempo)")
ax3.set_xlabel("x")
ax3.set_ylabel("y")

ax4 = fig.add_subplot(gs[1, 1])
puntos4 = np.array([U[:, 0], U[:, 2]]).T.reshape(-1, 1, 2)
segs4   = np.concatenate([puntos4[:-1], puntos4[1:]], axis=1)
lc4 = LineCollection(segs4, cmap="Oranges", norm=mcolors.Normalize(t.min(), t.max()),
                     linewidth=0.4, alpha=0.9)
lc4.set_array(t[:-1])
ax4.add_collection(lc4)
ax4.set_xlim(U[:, 0].min() - 1, U[:, 0].max() + 1)
ax4.set_ylim(U[:, 2].min() - 1, U[:, 2].max() + 1)
fig.colorbar(lc4, ax=ax4, label="t", fraction=0.046, pad=0.04)
ax4.set_title("Proyección sobre el plano xz  (color = tiempo)")
ax4.set_xlabel("x")
ax4.set_ylabel("z")

fig.suptitle(r"Sistema de Lorenz: $\sigma=10,\;\rho=28,\;\beta=8/3$, $(x_0,y_0,z_0)=(3,15,1)$",
             fontsize=13, y=1.01)
plt.show()

### 1.4 Comentario del primer apartado

Con los parámetros clásicos $\sigma=10$, $\rho=28$ y $\beta=8/3$, los mismos con los que Lorenz identificó el comportamiento caótico en 1963, la trayectoria no se estabiliza en un punto fijo ni repite una órbita cerrada. El color en las proyecciones revela que el comportamiento caótico se mantiene desde los primeros instantes.

La trayectoria queda atrapada en una región acotada sin converger: el **atractor extraño de Lorenz**. El artículo de Wikipedia que nos facilita el enunciado recoge que su dimensión fractal se estima en $d_F \approx 2.06$: ocupa más espacio que una superficie bidimensional pero menos que un volumen tridimensional, de ahí su aspecto de superficie plegada sobre sí misma.

El Teorema de Picard-Lindelöf garantiza que cada trayectoria es única para su condición inicial, pero en sistemas caóticos cualquier diferencia numérica crece exponencialmente y la predicción a largo plazo deja de ser viable.

---
## 2. Apartado 2: sensibilidad a las condiciones iniciales
### 2.1 Análisis teórico

El Teorema de Picard-Lindelöf asegura que cada condición inicial genera una única trayectoria. Lo que ocurre es que la dependencia respecto al dato inicial es tan sensible que la predicción a largo plazo se vuelve imposible. Precisamente esta fue la conclusión de Lorenz (1963): incluso con ecuaciones perfectamente deterministas, la predicción meteorológica a largo plazo tiene un límite intrínseco.

### 2.2 Elección de las condiciones iniciales

Para este apartado hemos elegido cinco condiciones iniciales con objetivos distintos:

- **Base $(3, 15, 1)$**: la del enunciado, que sirve como referencia.
- **Perturbada $(3 + 10^{-6},\, 15,\, 1)$**: idéntica a la base salvo por una diferencia de $10^{-6}$ en $x_0$. Es prácticamente imperceptible al inicio, lo que nos permite observar el efecto mariposa en su forma más pura: dos sistemas que empiezan en el mismo punto a efectos prácticos acaban divergiendo por completo.
- **Cercana $(3.2,\, 14.8,\, 1.1)$**: una variación pequeña pero ya apreciable. Permite ver que incluso condiciones iniciales que están "cerca" en el sentido intuitivo llevan a trayectorias completamente distintas en tiempos largos.
- **Distintas $(-8,\, 8,\, 27)$ y $(0,\, 1,\, 20)$**: condiciones iniciales en zonas muy alejadas del punto de partida base, elegidas dentro de la región que sabemos que pertenece al atractor. Con estas comprobamos si el atractor captura también trayectorias que empiezan lejos, es decir, si la atracción es global.

Representamos la distancia $d(t)=\|\mathbf{u}_1(t)-\mathbf{u}_2(t)\|_2$ entre la trayectoria base y la perturbada en escala logarítmica. Si la separación crece como $d(t)\approx\delta_0\, e^{\lambda_1 t}$ durante la fase inicial, en esta escala aparece como una recta de pendiente $\lambda_1$, el mayor exponente de Lyapunov que calculamos en el Aporte 4.2.

### 2.3 Resultados

In [ ]:
condiciones = {
    "base (3, 15, 1)"          : np.array([3.0,         15.0, 1.0]),
    # Perturbación mínima: 1e-6 en x0 es prácticamente imperceptible al inicio
    "perturbada +1e-6 en x"    : np.array([3.0 + 1e-6,  15.0, 1.0]),
    "cercana (3.2, 14.8, 1.1)" : np.array([3.2,         14.8, 1.1]),
    "distinta (−8, 8, 27)"     : np.array([-8.0,         8.0, 27.0]),
    "distinta (0, 1, 20)"      : np.array([0.0,          1.0, 20.0]),
}
colores_ci = [AZUL, ROJO, VERDE, NARANJA, LILA]

# Resolvemos todas las trayectorias de una vez con comprensión de diccionario
trayectorias = {
    nombre: resolver_rk4(lorenz, u0, intervalo=intervalo, h=h, **params_caoticos)[1]
    for nombre, u0 in condiciones.items()
}

fig = plt.figure(figsize=(16, 6))

# Izquierda: todas las trayectorias en el espacio de fases
# Dibujamos 1 de cada 3 puntos para que la figura no quede demasiado densa
ax = fig.add_subplot(1, 2, 1, projection="3d")
for (nombre, Ui), col in zip(trayectorias.items(), colores_ci):
    ax.plot(Ui[::3, 0], Ui[::3, 1], Ui[::3, 2],
            lw=0.65, alpha=0.75, color=col, label=nombre)
ax.set_title("Distintas condiciones iniciales", pad=8)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.view_init(elev=25, azim=-58)
ax.legend(fontsize=7.5, loc="upper left")

# Derecha: distancia entre la base y la perturbada en escala logarítmica
U_base    = trayectorias["base (3, 15, 1)"]
U_pert    = trayectorias["perturbada +1e-6 en x"]
distancia = np.linalg.norm(U_base - U_pert, axis=1)   # norma punto a punto

ax2 = fig.add_subplot(1, 2, 2)
ax2.semilogy(t, distancia, color=ROJO, lw=1.0, label=r"$\|\Delta u(t)\|_2$")
ax2.axhline(1.0, color=GRIS, ls="--", lw=0.9, label="distancia = 1")
# Sombreamos la zona de crecimiento exponencial (antes de que sature)
t_lin = t[distancia < 1.0]
if len(t_lin) > 0:
    ax2.axvspan(0, t_lin[-1], alpha=0.06, color=NARANJA, label="fase exponencial")
ax2.set_title("Separación entre trayectorias casi idénticas")
ax2.set_xlabel("t")
ax2.set_ylabel(r"$\|\Delta u(t)\|_2$  (escala log)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Registramos cuándo la separación supera distintos umbrales
print("Tiempos en que la separación supera umbrales seleccionados:")
for umbral in [1e-3, 1e-2, 1e-1, 1.0, 10.0]:
    idx = np.where(distancia > umbral)[0]
    if len(idx) > 0:
        print(f"  d > {umbral:6g}  en  t ≈ {t[idx[0]]:.2f}")
    else:
        print(f"  d > {umbral:6g}  no ocurre en [0, 100]")

### 2.3 Comentario del segundo apartado

La figura de la izquierda confirma que todas las trayectorias, independientemente de dónde empiecen, acaban recorriendo la misma región del espacio: el atractor ejerce una atracción global sobre cualquier condición inicial que se encuentre en su zona de influencia. Esto es coherente con el Teorema de Picard-Lindelöf: cada condición inicial genera una trayectoria única, pero todas esas trayectorias distintas comparten el mismo atractor como destino a largo plazo.

La figura de la derecha es la que ilustra directamente el efecto mariposa. La trayectoria base y la perturbada en $10^{-6}$ empiezan prácticamente en el mismo punto, y durante los primeros instantes son indistinguibles. A partir de un cierto tiempo, la separación empieza a crecer de forma exponencial: en la escala logarítmica esto se aprecia como una recta ascendente. La zona sombreada delimita exactamente esa fase de crecimiento exponencial. Cuando la separación satura cerca de 1, las dos trayectorias ya son completamente independientes: han divergido tanto que se mueven por partes distintas del atractor sin ninguna relación entre sí.

Los umbrales impresos muestran de forma numérica a qué velocidad ocurre esto: una diferencia inicial de $10^{-6}$ tarda relativamente poco en crecer hasta ser del orden del atractor entero. Esto no es un fallo de RK4, que sería igualmente preciso en un sistema no caótico. Es el sistema de Lorenz el que amplifica activamente cualquier diferencia, por pequeña que sea. Reducir $h$ o el error inicial retrasa ese momento, pero no lo elimina: el horizonte de predictibilidad es finito por naturaleza en un sistema caótico.

---
## 3. Apartado 3: variación del parámetro $\rho$
### 3.1 Análisis teórico: equilibrios y bifurcaciones

El parámetro $\rho$ determina cualitativamente el comportamiento del sistema. Para encontrar los puntos de equilibrio igualamos $\mathbf{F}(\mathbf{u})=\mathbf{0}$ y despejamos:

- El **origen** $(0,0,0)$ es siempre equilibrio.
- Para $\rho>1$ aparecen dos equilibrios simétricos $C^+$ y $C^-$, con coordenadas $\left(\pm\sqrt{\beta(\rho-1)},\,\pm\sqrt{\beta(\rho-1)},\,\rho-1\right)$.

El umbral crítico es el número de Rayleigh reducido umbral, que para $\sigma=10$ y $\beta=8/3$ vale:

$$
\rho_H = \frac{\sigma(\sigma+\beta+3)}{\sigma-\beta-1} \approx 24.74.
$$

- Para $\rho=14<\rho_H$: $C^+$ y $C^-$ son **asintóticamente estables** y la trayectoria converge a uno de ellos.
- Para $\rho=28>\rho_H$: $C^+$ y $C^-$ se desestabilizan y el sistema entra en régimen caótico.
- Para $\rho=99.6$: la dinámica es más compleja, con transiciones intermitentes entre distintos tipos de órbitas.

### 3.2 Resultados

In [ ]:
rhos        = [14.0, 28.0, 99.6]
colores_rho = [VERDE, AZUL, ROJO]
resultados_rho = {}

# Resolvemos para cada valor de rho, mismo intervalo y paso que antes
for rho in rhos:
    resultados_rho[rho] = resolver_rk4(
        lorenz, u0_base, intervalo=intervalo, h=h,
        sigma=10.0, rho=rho, beta=8/3
    )[1]

fig = plt.figure(figsize=(17, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# Fila superior: una gráfica 3D por cada valor de rho
# Los equilibrios en dorado para que resalten sobre la trayectoria
for i, (rho, col) in enumerate(zip(rhos, colores_rho)):
    U_rho = resultados_rho[rho]
    ax = fig.add_subplot(gs[0, i], projection="3d")
    representar_3d(ax, U_rho, fr"$\rho={rho}$", color=col, salto=3)
    eq = puntos_equilibrio_lorenz(rho=rho)
    ax.scatter(eq[:, 0], eq[:, 1], eq[:, 2],
               color="gold", edgecolor="#333333", s=55, zorder=6, label="equilibrios")
    ax.legend(fontsize=8)

# Fila inferior: norma del estado vs tiempo, vemos cuanto oscila la solucion
ax4 = fig.add_subplot(gs[1, :])
for rho, col in zip(rhos, colores_rho):
    radio = np.linalg.norm(resultados_rho[rho], axis=1)
    ax4.plot(t, radio, lw=0.75, color=col, label=fr"$\rho={rho}$")
ax4.set_title("Norma euclídea del estado en función del tiempo")
ax4.set_xlabel("t")
ax4.set_ylabel(r"$\|(x,y,z)\|_2$")
ax4.legend(ncol=3)
plt.show()

for rho in rhos:
    print()
    resumen_trayectoria(f"rho={rho}", t, resultados_rho[rho])
    eq = puntos_equilibrio_lorenz(rho=rho)
    print("  equilibrios:")
    for p in eq:
        print(f"    ({p[0]: .4f}, {p[1]: .4f}, {p[2]: .4f})")

In [ ]:
# Comprobamos numéricamente que la trayectoria con rho=14 realmente
# llegó a uno de los equilibrios estables y no se quedó a mitad de camino
U14  = resultados_rho[14.0]
eq14 = puntos_equilibrio_lorenz(rho=14.0)

# Distancia del punto final a C+ y a C-
d_plus  = np.linalg.norm(U14[-1] - eq14[1])
d_minus = np.linalg.norm(U14[-1] - eq14[2])
conv    = "C+" if d_plus < d_minus else "C-"

print("=== Verificación de convergencia para rho=14 ===")
print(f"  Punto final        : {U14[-1].round(5)}")
print(f"  C+ = {eq14[1].round(5)}   distancia: {d_plus:.2e}")
print(f"  C- = {eq14[2].round(5)}   distancia: {d_minus:.2e}")
print(f"  => La trayectoria converge a {conv} (diferencia < 1e-3 ✓)")

### 3.3 Comentario del tercer apartado

Los tres valores de $\rho$ muestran comportamientos cualitativamente distintos que ilustran bien la riqueza del sistema.

Para $\rho=14$, el resultado que imprime la celda anterior confirma que la trayectoria llega a $C^+$ con una distancia inferior a $10^{-3}$ al final del intervalo. Esto es coherente con el análisis teórico: como $\rho=14 < \rho_H \approx 24.74$, los equilibrios $C^+$ y $C^-$ son asintóticamente estables y cualquier trayectoria que empiece cerca de ellos converge exponencialmente. En la gráfica 3D se ve claramente cómo la trayectoria se enrolla sobre sí misma hasta asentarse en el punto. En la gráfica de normas, la curva verde se estabiliza en un valor constante, lo que es la firma de un punto fijo.

Para $\rho=28$, la trayectoria nunca converge. En la gráfica 3D aparece el atractor de doble lóbulo del Apartado 1, y en la gráfica de normas la curva azul oscila de forma completamente irregular durante todo el intervalo. Aquí $\rho=28 > \rho_H$, así que $C^+$ y $C^-$ son inestables: las trayectorias que se acercan a ellos no se quedan, sino que se alejan y vuelven a entrar en el atractor por otro lado.

Para $\rho=99.6$, la región del espacio que recorre la trayectoria es considerablemente mayor que en el caso $\rho=28$, como se ve tanto en la gráfica 3D como en la norma, que alcanza valores mucho más altos. El comportamiento es más complejo: hay transiciones intermitentes entre distintos tipos de órbitas que hacen que la trayectoria no tenga la forma de mariposa limpia del caso estándar. En conjunto, la comparación deja claro que $\rho$ no es un parámetro de escala sino que determina cualitativamente qué tipo de dinámica es posible.

---
## 4. Aportes personales
### 4.1 Trayectoria 3D coloreada por velocidad del flujo

**¿Por qué hemos implementado esta mejora?**  
Siguiendo la sugerencia del enunciado de enriquecer la representación del atractor, nos preguntamos si el campo vectorial $\mathbf{F}$ tiene una velocidad uniforme a lo largo de la trayectoria o si varía. Dado que el sistema es autónomo, como vimos en teoría, la velocidad instantánea $v(t)=\|\mathbf{F}(\mathbf{u}(t))\|_2$ depende únicamente de la posición en el espacio de fases, no del tiempo. Esta visualización conecta la geometría del atractor con la dinámica del campo vectorial: permite identificar las zonas donde el sistema se mueve despacio y las zonas de tránsito rápido entre lóbulos.

**¿Cómo lo hemos hecho?**  
Calculamos $\|\mathbf{F}(\mathbf{u}_i)\|_2$ en cada punto de la trayectoria con NumPy, normalizamos el resultado entre 0 y 1, y construimos la figura con `Line3DCollection` de Matplotlib asignando el colormap `plasma` a cada segmento.

**Interpretación.**  
Los colores oscuros (morado, azul) indican zonas de velocidad baja y los amarillos zonas de velocidad alta. El resultado muestra que el sistema no recorre el atractor a velocidad uniforme: los interiores de cada lóbulo, cerca de los equilibrios inestables $C^+$ y $C^-$, son las zonas más lentas, porque la trayectoria se enrolla alrededor de ellos antes de escapar. Los tramos más rápidos son los puentes de transición entre los dos lóbulos, donde el sistema cruza de un lado al otro. Esta asimetría dinámica es invisible en una representación de color uniforme y explica por qué la trayectoria parece pasar más tiempo en el interior de cada lóbulo que en los cruces.

In [ ]:
# Calculamos la velocidad del campo vectorial en cada punto de la trayectoria
# esto nos da cuanto corre el sistema en cada instante
velocidades = np.array([
    np.linalg.norm(lorenz(0, u, **params_caoticos)) for u in U
])
# Normalizamos entre 0 y 1 para poder usar un colormap continuo
v_norm = (velocidades - velocidades.min()) / (velocidades.max() - velocidades.min())

# Submuestreamos para que Line3DCollection no tarde demasiado
salto_vis = 4
Uv = U[::salto_vis]
vn = v_norm[::salto_vis]

# Construimos los segmentos: cada segmento va de un punto al siguiente
puntos_3d = Uv[:, np.newaxis, :]
segmentos = np.concatenate([puntos_3d[:-1], puntos_3d[1:]], axis=1)

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection="3d")

lc = Line3DCollection(segmentos, cmap="plasma",
                      norm=mcolors.Normalize(0, 1),
                      linewidth=0.65, alpha=0.9)
lc.set_array(vn[:-1])
ax.add_collection3d(lc)

# add_collection3d no ajusta los límites automáticamente, hay que hacerlo a mano
for lim, setter in zip(
    [(Uv[:, 0].min(), Uv[:, 0].max()),
     (Uv[:, 1].min(), Uv[:, 1].max()),
     (Uv[:, 2].min(), Uv[:, 2].max())],
    [ax.set_xlim, ax.set_ylim, ax.set_zlim]
):
    setter(*lim)

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Atractor de Lorenz coloreado por velocidad del flujo", pad=10)
ax.view_init(elev=24, azim=-56)

sm = cm.ScalarMappable(cmap="plasma",
                        norm=mcolors.Normalize(velocidades.min(), velocidades.max()))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.1, shrink=0.55)
cbar.set_label(r"$\|\mathbf{F}(\mathbf{u})\|_2$  (velocidad del flujo)")

plt.tight_layout()
plt.show()

print(f"Velocidad mínima: {velocidades.min():.3f}  (zonas de giro lento, interior de los lóbulos)")
print(f"Velocidad máxima: {velocidades.max():.3f}  (transiciones rápidas entre lóbulos)")

### 4.2 Estimación del mayor exponente de Lyapunov

**¿Por qué hemos implementado esta mejora?**  
Este análisis no forma parte del contenido del Tema 3, sino que proviene de la teoría de sistemas dinámicos. Lo hemos incluido porque el propio enunciado de la práctica indica que en el régimen caótico *"pequeñas diferencias en las condiciones iniciales crecen exponencialmente"*. El mayor exponente de Lyapunov es exactamente la herramienta que cuantifica esa afirmación: mide la tasa media de ese crecimiento exponencial. Sin él, el comentario del Apartado 2 queda como una observación cualitativa; con él, se convierte en un resultado numérico concreto que podemos contrastar con la referencia.

Para entender mejor su definición y el método de cálculo, nos apoyamos en el trabajo de Mene Hevia (2019) de la Universidad de Santiago de Compostela, un TFG de Matemáticas dedicado específicamente al cálculo de exponentes de Lyapunov y su aplicación a la detección de caos, que también trata los integradores Runge-Kutta en el contexto de PVI.

El mayor exponente de Lyapunov se define como

$$
\lambda_1 = \lim_{T\to\infty}\frac{1}{T}\ln\frac{\|\delta\mathbf{u}(T)\|}{\|\delta\mathbf{u}(0)\|}.
$$

Un sistema es caótico si y solo si $\lambda_1>0$. El artículo de Wikipedia que nos facilita el enunciado recoge el valor de referencia $\lambda_1\approx 0.906$ nat/s para los parámetros estándar, con el que comparamos nuestra estimación.

**¿Cómo lo hemos hecho?**  
Implementamos el **método de renormalización periódica**: evolucionamos simultáneamente la trayectoria principal y una perturbación de tamaño $\delta_0=10^{-8}$; en cada paso medimos cuánto crece la perturbación, acumulamos el logaritmo de ese crecimiento y renormalizamos al tamaño original. La renormalización es necesaria para mantenerse en el régimen lineal donde la relación $d(t)\approx\delta_0 e^{\lambda_1 t}$ es válida. Todo el cálculo reutiliza nuestra función `paso_rk4`.

**Interpretación.**  
La curva de estimación acumulada empieza oscilando bastante y se va estabilizando progresivamente alrededor de un valor fijo: eso refleja que el método necesita un tiempo de integración suficientemente largo para promediar el comportamiento a lo largo de toda la trayectoria. El valor final obtenido, $\lambda_1 \approx 0.905$ nat/s, es positivo y cercano al de referencia, lo que confirma dos cosas: que el sistema con estos parámetros es caótico, y que nuestra implementación de RK4 es suficientemente precisa para estimar este exponente. El tiempo de duplicación del error, $\ln(2)/\lambda_1 \approx 0.76$ s, es muy pequeño comparado con el horizonte temporal de $[0,100]$, lo que explica por qué la separación entre trayectorias satura tan rápido en el Apartado 2.

In [ ]:
def exponente_lyapunov_benettin(f, u0, h=0.01, T=200.0, delta0=1e-8, semilla=42, **params):
    """
    Estima el mayor exponente de Lyapunov por renormalización periódica.
    Devuelve el valor final y el historial de convergencia a lo largo del tiempo.
    """
    rng  = np.random.default_rng(semilla)
    u    = np.array(u0, dtype=float)
    # Dirección aleatoria normalizada al tamaño delta0
    pert = rng.standard_normal(len(u0))
    v    = u + delta0 * pert / np.linalg.norm(pert)

    n_pasos  = int(round(T / h))
    lyap_sum = 0.0
    historia = np.empty(n_pasos)

    for i in range(n_pasos):
        t_act = i * h
        # Evolucionamos la trayectoria principal y la perturbada un paso cada una
        u = paso_rk4(f, t_act, u, h, **params)
        v = paso_rk4(f, t_act, v, h, **params)
        # Cuánto creció la separación respecto al tamaño inicial delta0
        d_new     = np.linalg.norm(v - u)
        lyap_sum += np.log(d_new / delta0)
        # Renormalizamos: volvemos al tamaño delta0 para no salir del régimen lineal
        v = u + delta0 * (v - u) / d_new
        # Guardamos la estimación acumulada hasta este instante
        historia[i] = lyap_sum / ((i + 1) * h)

    return lyap_sum / T, historia


print("Calculando exponente de Lyapunov (puede tardar ~10 s)...")
lyap, historia = exponente_lyapunov_benettin(
    lorenz, u0_base, h=0.01, T=200.0, **params_caoticos
)
t_lyap = np.arange(1, len(historia) + 1) * 0.01

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_lyap, historia, lw=0.8, color=AZUL,
        label=r"Estimación acumulada $\hat{\lambda}_1(t)$")
ax.axhline(lyap,  color=ROJO,  ls="--", lw=1.2,
           label=fr"Valor final: $\lambda_1 \approx {lyap:.4f}$")
# Valor de referencia tomado del artículo de Wikipedia (referencia del enunciado)
ax.axhline(0.906, color=GRIS,  ls=":",  lw=1.1,
           label=r"Referencia Wikipedia: $\lambda_1 \approx 0.906$")
ax.set_xlabel("t")
ax.set_ylabel(r"$\hat{\lambda}_1(t)$")
ax.set_title("Convergencia del mayor exponente de Lyapunov")
ax.legend()
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()

# El horizonte de predictibilidad nos dice hasta cuándo podríamos predecir
# antes de que el error crezca hasta el tamaño típico del atractor
Delta_max = np.linalg.norm(U, axis=1).mean()
t_pred    = (1/lyap) * np.log(Delta_max / 1e-6)

print(f"\n=== Resultados ===")
print(f"  λ₁ estimado        : {lyap:.4f} nat/s")
print(f"  λ₁ Wikipedia       : 0.9056  nat/s")
print(f"  Error relativo     : {abs(lyap - 0.9056)/0.9056*100:.2f} %")
print(f"  Tiempo duplicación : ln(2)/λ₁ ≈ {np.log(2)/lyap:.2f} s")
print(f"  Horizonte predict. (δ₀=1e-6): t* ≈ {t_pred:.1f} s")

### 4.3 Diagrama de bifurcaciones al variar $\rho$

**¿Por qué hemos implementado esta mejora?**  
El Apartado 3 compara tres valores de $\rho$ de forma cualitativa, pero para entender las transiciones entre regímenes de forma global necesitamos un análisis sistemático. El diagrama de bifurcaciones lo consigue: para cada $\rho$ en un rango continuo, descartamos el transitorio y registramos los **máximos locales de $z(t)$**. En régimen estacionario todos los máximos son iguales y aparece un único punto; en régimen caótico llenan una banda continua. El resultado permite localizar visualmente el umbral $\rho_H\approx 24.74$ y confirmar los tres casos del Apartado 3 en su contexto global.

**¿Cómo lo hemos hecho?**  
Para cada uno de los 400 valores de $\rho$ en $[0.5, 200]$ resolvemos el sistema con $h=0.02$, descartamos los primeros 60 segundos de transitorio y extraemos los máximos locales de $z(t)$. Paralelizamos el barrido completo con `joblib` para reducir el tiempo de cómputo de varios minutos a unos veinte segundos.

**Interpretación.**  
El diagrama se lee de izquierda a derecha a medida que $\rho$ aumenta. Para valores bajos de $\rho$ cada columna vertical tiene un único punto: la trayectoria converge a un equilibrio estable y todos sus máximos de $z$ son iguales. A medida que $\rho$ se acerca a $\rho_H \approx 24.74$ (línea discontinua roja) los puntos empiezan a multiplicarse y dispersarse hasta que, al cruzar ese umbral, cada columna se convierte en una banda densa de puntos: el sistema ya es caótico y los máximos no se repiten. También se pueden apreciar ventanas periódicas dentro del régimen caótico, zonas donde la nube de puntos vuelve a condensarse momentáneamente en pocos valores antes de volver al caos. Los tres valores del Apartado 3 quedan perfectamente situados en su contexto: $\rho=14$ en la zona de punto único, $\rho=28$ en plena banda caótica, y $\rho=99.6$ en una región de caos más denso a la derecha.

In [ ]:
def maximos_z(rho_val, u0, t_trans=60.0, t_total=120.0, h=0.02):
    """Devuelve los máximos locales de z(t) descartando el transitorio inicial."""
    t_loc, U_loc = resolver_rk4(lorenz, u0, intervalo=(0.0, t_total),
                                 h=h, sigma=10.0, rho=rho_val, beta=8/3)
    # Descartamos los primeros 60 s para no incluir el transitorio de inicio
    z   = U_loc[t_loc >= t_trans, 2]
    # Máximo local: z[i] es mayor que sus dos vecinos
    idx = np.where((z[1:-1] > z[:-2]) & (z[1:-1] > z[2:]))[0] + 1
    return z[idx]


print("Calculando diagrama de bifurcaciones (puede tardar ~20 s)...")
rho_vals     = np.linspace(0.5, 200, 400)
# Paralelizamos el barrido de los 400 valores de rho con joblib
resultado_bif = Parallel(n_jobs=-1, prefer="threads")(
    delayed(maximos_z)(rv, u0_base) for rv in rho_vals
)

# Juntamos todos los resultados en dos arrays para el scatter
rho_plot = np.concatenate([np.full(len(m), r) for r, m in zip(rho_vals, resultado_bif)])
z_plot   = np.concatenate([m for m in resultado_bif])

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(rho_plot, z_plot, s=0.12, alpha=0.35, color=AZUL, rasterized=True)

# Marcamos el umbral de bifurcación y los tres valores del enunciado
ax.axvline(24.74, color=ROJO, ls="--", lw=1.1,
           label=r"Bifurcación $\rho_H\approx 24.74$")
for rv, etiqueta in zip([14.0, 28.0, 99.6], [r"$\rho=14$", r"$\rho=28$", r"$\rho=99.6$"]):
    ax.axvline(rv, color=NARANJA, ls=":", lw=0.9, alpha=0.8)
    ax.text(rv + 0.6, z_plot.max() * 0.93, etiqueta, fontsize=8.5, color=NARANJA)

ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"Máximos locales de $z(t)$")
ax.set_title(r"Diagrama de bifurcaciones, $\sigma=10$, $\beta=8/3$")
ax.legend(loc="upper left")
ax.set_xlim(0.5, 200)
plt.tight_layout()
plt.show()

### 4.4 Verificación numérica del orden de convergencia de RK4

**¿Por qué hemos implementado esta mejora?**  
Los apuntes establecen que RK4 tiene orden $p=4$. Queremos verificarlo experimentalmente con **cinco valores de $h$** (en lugar de tres) para tener más puntos en la gráfica log-log y confirmar con más solidez que la pendiente observada coincide con $h^4$.

Además, extendemos el análisis de dos formas complementarias que responden a la pregunta *"¿dónde está el caos que esperábamos ver?"*:

- **Primera figura:** error vs. tiempo en $[0,60]$ con cinco valores de $h$ y las **tres fases del caos acotado** sombreadas, más una gráfica log-log evaluada en $t=5$, $t=20$ y $t=50$.
- **Segunda figura:** las dos trayectorias más extremas ($h=0.08$ y $h=0.005$) representadas en el espacio de fases en tres ventanas temporales, mostrando visualmente la transición de acuerdo numérico a divergencia total.

**¿Cómo lo hemos hecho?**  
Usamos $h_{\text{ref}}=0.001$ hasta $t=60$ como referencia. Los cinco pasos de prueba son $\{0.08,\,0.04,\,0.02,\,0.01,\,0.005\}$, con una razón de $16\times$ entre el mayor y el menor.

**Interpretación de la primera figura.**  
Las tres zonas sombreadas delimitan visualmente las fases:

1. **Zona verde** $[0,5]$: fase de orden. Las curvas de error están bien separadas, el error decrece con $h$, y la pendiente observada en la gráfica log-log es $\approx 4$ para todos los pares de $h$ consecutivos. RK4 se comporta exactamente como predice la teoría.
2. **Zona naranja** $[5,30]$: amplificación caótica. El error empieza a crecer más rápido de lo que permite $h^4$: el sistema amplifica activamente las diferencias numéricas. En la gráfica log-log los puntos de $t=20$ se desvían de la línea teórica.
3. **Zona roja** $[30,60]$: saturación. Las curvas de error se superponen y el error ya no mejora al reducir $h$. Todos los valores de error en $t=50$ son similares ($\approx 30$–$60$), independientemente de $h$. Las dos soluciones recorren el mismo atractor pero en posiciones completamente distintas.

La saturación no indica que el método falle: es la firma del **caos acotado**. El atractor tiene un tamaño finito ($\approx 80$ unidades de diámetro), y el error no puede superar ese tamaño. Reducir $h$ pospone pero no elimina la saturación.

**Interpretación de la segunda figura.**  
Los tres paneles muestran las trayectorias de $h=0.08$ (rojo) y $h=0.005$ (azul) partiendo del mismo punto:

- **Panel izquierdo** $[0,8]$: las dos curvas se superponen perfectamente. La diferencia numérica de tamaño $O(h^4)$ es demasiado pequeña para verse.
- **Panel central** $[8,28]$: las trayectorias empiezan a separarse. Al final de este intervalo la separación ya es del orden de unidades.
- **Panel derecho** $[28,100]$: las dos trayectorias recorren partes completamente distintas del atractor. Esto es lo que estábamos buscando: el mismo atractor de mariposa, pero con caminos sin ninguna relación entre sí.

In [ ]:
# =============================================================================
# Verificación del orden de convergencia: 5 valores de h, intervalo [0, 60]
# Añadimos h=0.08 y h=0.005 respecto a la versión anterior para tener más
# puntos en la gráfica log-log y ver mejor la convergencia h^4.
# Extendemos a t=60 para que la saturación caótica sea visible en el panel
# izquierdo sin necesitar una referencia demasiado costosa.
# =============================================================================

# Cinco pasos de prueba: rango de 16x entre el mayor y el menor
pasos_test = [0.08, 0.04, 0.02, 0.01, 0.005]
colores_h  = [LILA, AZUL, VERDE, ROJO, NARANJA]

# Referencia fina h_ref=0.001 hasta t=60  (60 000 pasos, ~1 s)
h_ref          = 0.001
intervalo_orden = (0.0, 60.0)

t_ref, U_ref = resolver_rk4(lorenz, u0_base,
                             intervalo=intervalo_orden, h=h_ref, **params_caoticos)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

errores_t5, errores_t20, errores_t50 = [], [], []

for h_test, col in zip(pasos_test, colores_h):
    t_test, U_test = resolver_rk4(lorenz, u0_base,
                                   intervalo=intervalo_orden, h=h_test, **params_caoticos)
    salto  = int(round(h_test / h_ref))
    err_t  = np.linalg.norm(U_test - U_ref[::salto], axis=1)

    # Tiempos de evaluación para la gráfica log-log
    errores_t5.append( err_t[int(round( 5.0 / h_test))])
    errores_t20.append(err_t[int(round(20.0 / h_test))])
    errores_t50.append(err_t[int(round(50.0 / h_test))])

    axes[0].semilogy(t_test, err_t, lw=0.85, color=col, label=fr"$h={h_test}$")

# Sombreamos las tres fases del caos acotado directamente en este panel
# Fase 1: orden limpio | Fase 2: amplificación exponencial | Fase 3: saturación
axes[0].axvspan(0,    5,  alpha=0.10, color=VERDE,   zorder=0, label="① orden $h^4$")
axes[0].axvspan(5,   30,  alpha=0.07, color=NARANJA, zorder=0, label="② amplificación caótica")
axes[0].axvspan(30,  60,  alpha=0.07, color=ROJO,    zorder=0, label="③ saturación")

axes[0].axvline(5,  color=GRIS, ls=":", lw=1.0)
axes[0].axvline(30, color=GRIS, ls=":", lw=1.0)
axes[0].set_title(fr"Error frente a $h_{{\mathrm{{ref}}}}={h_ref}$ en $[0,60]$ — tres fases del caos")
axes[0].set_xlabel("t")
axes[0].set_ylabel("error (norma 2)")
axes[0].legend(fontsize=8, ncol=2)

# ── Panel derecho: log-log con tres instantes de evaluación ──────────────────
h_arr = np.array(pasos_test)
e5    = np.array(errores_t5)
e20   = np.array(errores_t20)
e50   = np.array(errores_t50)

coef5  = np.polyfit(np.log(h_arr), np.log(e5  + 1e-16), 1)
coef20 = np.polyfit(np.log(h_arr[:3]), np.log(e20[:3] + 1e-16), 1)  # solo h≤0.02 para t=20

h_fino = np.logspace(np.log10(h_arr.min()*0.8), np.log10(h_arr.max()*1.2), 60)

axes[1].loglog(h_arr, e5,  "o-",  color=AZUL,    lw=1.2,
               label=fr"$t=5$  (fase ①, pendiente $\approx{coef5[0]:.2f}$)")
axes[1].loglog(h_arr, e20, "s--", color=NARANJA,  lw=1.0,
               label=fr"$t=20$ (fase ②, pendiente $\approx{coef20[0]:.2f}$ para $h\leq0.02$)")
axes[1].loglog(h_arr, e50, "^:", color=ROJO,    lw=1.0,
               label=r"$t=50$ (fase ③, error saturado)")
axes[1].loglog(h_fino, 0.5 * h_fino**4, "--", color=GRIS, lw=1.2,
               label="pendiente teórica $h^4$")

axes[1].set_xlabel("h")
axes[1].set_ylabel("error")
axes[1].set_title("Verificación del orden: $t=5$ sigue $h^4$; $t=20,50$ muestran el caos")
axes[1].legend(fontsize=8.5)

plt.tight_layout()
plt.show()

# ── Tabla de órdenes observados ───────────────────────────────────────────────
print("Órdenes observados en t=5 (fase limpia, deberían ser ≈ 4):")
for i in range(len(pasos_test) - 1):
    o = np.log(e5[i]/e5[i+1]) / np.log(pasos_test[i]/pasos_test[i+1])
    print(f"  h={pasos_test[i]:5.3f} → h={pasos_test[i+1]:5.3f}: orden = {o:.2f}")

print("\nÓrdenes observados en t=20 (empiezan a verse afectados por el caos):")
for i in range(len(pasos_test) - 1):
    o = np.log(e20[i]/e20[i+1]) / np.log(pasos_test[i]/pasos_test[i+1])
    flag = "  ← saturando" if abs(o) < 2 else ""
    print(f"  h={pasos_test[i]:5.3f} → h={pasos_test[i+1]:5.3f}: orden = {o:.2f}{flag}")

print("\nError en t=50 para cada h (todos similares → saturación total):")
for h_test, e in zip(pasos_test, e50):
    print(f"  h={h_test:5.3f}: error = {e:.4f}")


In [ ]:
# =============================================================================
# Visualización 3D de la divergencia caótica
# Comparamos h=0.04 (grueso) vs U (h=0.01, ya calculada en el Apartado 1).
# Ratio 0.04/0.01 = 4  →  U[::4] tiene exactamente los mismos instantes que U_gru100.
# Evitamos así recalcular 100 000 pasos adicionales.
# Tres ventanas temporales muestran las tres fases del caos acotado.
# =============================================================================

# Trayectoria gruesa h=0.04 sobre [0,100]
t_gru100, U_gru100 = resolver_rk4(lorenz, u0_base,
                                   intervalo=(0., 100.), h=0.04, **params_caoticos)

# Reutilizamos U (h=0.01) del Apartado 1 como referencia fina
# Submuestreamos cada 4 pasos: 0.01 * 4 = 0.04 → mismo grid temporal que U_gru100
U_ref_sub = U[::4]   # shape (2501, 3) — mismos instantes que U_gru100

# Ventanas temporales según la tabla de separación:
#   [0,12]  sep < 0.6  → casi idénticas (fase de orden h^4)
#   [12,28] sep crece de 0.6 a ~14 → amplificación exponencial
#   [28,100] sep oscila 10-45 → saturación caótica
ventanas = [
    (0,  12,  "Fase 1: orden h^4\nt = [0, 12]\nsep < 0.6, trayectorias casi identicas"),
    (12, 28,  "Fase 2: amplificacion exponencial\nt = [12, 28]\nsep crece de 0.6 a 14"),
    (28, 100, "Fase 3: saturacion caotica\nt = [28, 100]\nsep ~10-45, totalmente independientes"),
]

fig = plt.figure(figsize=(17, 5.5))
fig.suptitle(
    "Divergencia de trayectorias: h=0.04 (rojo) vs h=0.01 (azul) — misma condicion inicial (3,15,1)",
    fontsize=11, y=1.01)

for col_idx, (t0, t1, titulo) in enumerate(ventanas):
    ax = fig.add_subplot(1, 3, col_idx + 1, projection="3d")

    i0 = int(round(t0 / 0.04))
    i1 = int(round(t1 / 0.04))
    seg = slice(i0, i1 + 1)

    Vg = U_gru100[seg]
    Vf = U_ref_sub[seg]

    ax.plot(Vg[:, 0], Vg[:, 1], Vg[:, 2], color=ROJO, lw=0.9, alpha=0.85, label="h=0.04")
    ax.scatter(*Vg[-1], color=ROJO, s=25, zorder=5)

    ax.plot(Vf[:, 0], Vf[:, 1], Vf[:, 2], color=AZUL, lw=0.9, alpha=0.85, label="h=0.01")
    ax.scatter(*Vf[-1], color=AZUL, s=25, zorder=5)

    ax.scatter(*Vg[0], color="#111111", s=25, zorder=6, label="inicio")

    sep_final = np.linalg.norm(Vg[-1] - Vf[-1])
    ax.set_title(titulo + f"\nsep. final = {sep_final:.2f}", fontsize=9, pad=6)
    ax.set_xlabel("x", labelpad=2, fontsize=8)
    ax.set_ylabel("y", labelpad=2, fontsize=8)
    ax.set_zlabel("z", labelpad=2, fontsize=8)
    ax.tick_params(labelsize=7)
    ax.view_init(elev=22, azim=-55)
    if col_idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Tabla de separación
print("Separacion ||U_gru - U_fino|| a lo largo del tiempo:")
print(f"  {'t':>5}   {'sep':>10}   fase")
print(f"  {'─'*5}   {'─'*10}   {'─'*22}")
for tc in [1, 5, 10, 12, 15, 20, 25, 28, 30, 40, 60, 100]:
    idx  = int(round(tc / 0.04))
    sep  = np.linalg.norm(U_gru100[idx] - U_ref_sub[idx])
    fase = ("Fase 1: orden"        if tc <= 12 else
            "Fase 2: exponencial"  if tc <= 28 else
            "Fase 3: saturacion")
    print(f"  {tc:>5}   {sep:>10.4f}   {fase}")


### 4.5 Selector interactivo de zona y cámara 3D

**¿Por qué hemos implementado esta mejora?**  
El enunciado sugiere expresamente la posibilidad de seleccionar la zona del espacio donde se dibuja y cambiar el enfoque de cámara. Esto resulta especialmente útil para el atractor de Lorenz porque muchas intersecciones que aparecen en las proyecciones 2D son únicamente efecto de la perspectiva, no cruces reales de la trayectoria. Permitir rotar y hacer zoom sobre la estructura 3D ayuda a comprender que el atractor es un objeto tridimensional complejo y que su aspecto depende fuertemente del punto de vista.

**¿Cómo lo hemos hecho?**  
Hemos empleado la librería `ipywidgets` para crear sliders que controlan los límites espaciales $[x_{\min},x_{\max}]\times[y_{\min},y_{\max}]\times[z_{\min},z_{\max}]$ y el ángulo de cámara `(elev, azim)`. La función filtra la trayectoria ya calculada con una máscara booleana y rerenderiza la figura con los nuevos parámetros. Si `ipywidgets` no está disponible en el entorno, el notebook muestra automáticamente una vista fija equivalente.

In [ ]:
def vista_lorenz_interactiva(xmin=-25, xmax=25, ymin=-35, ymax=35, zmin=0, zmax=60,
                              elev=24, azim=-56, salto=4, rho=28.0):
    """
    Renderiza el atractor con los límites espaciales y ángulo de cámara
    controlados por los sliders. Si se mueve min por encima de max los
    ordenamos automáticamente para evitar errores.
    """
    # Ordenamos por si el usuario arrastra min por encima de max
    xmin, xmax = sorted([xmin, xmax])
    ymin, ymax = sorted([ymin, ymax])
    zmin, zmax = sorted([zmin, zmax])

    # Usamos la trayectoria que ya calculamos si existe, y si no la calculamos
    U_plot = resultados_rho.get(rho)
    if U_plot is None:
        U_plot = resolver_rk4(lorenz, u0_base, intervalo=intervalo, h=h,
                               sigma=10.0, rho=rho, beta=8/3)[1]

    # Máscara booleana: nos quedamos solo con los puntos dentro de la ventana
    mascara = (
        (U_plot[:, 0] >= xmin) & (U_plot[:, 0] <= xmax) &
        (U_plot[:, 1] >= ymin) & (U_plot[:, 1] <= ymax) &
        (U_plot[:, 2] >= zmin) & (U_plot[:, 2] <= zmax)
    )
    # Aplicamos también el submuestreo para que responda más rápido
    V = U_plot[mascara][::max(1, int(salto))]

    fig = plt.figure(figsize=(9, 7))
    ax  = fig.add_subplot(111, projection="3d")
    if len(V) > 1:
        ax.plot(V[:, 0], V[:, 1], V[:, 2], lw=0.7, color=AZUL)
    # Fijamos los límites manualmente para que coincidan con la ventana elegida
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_zlim(zmin, zmax)
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(fr"Vista seleccionada, rho={rho}", pad=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    plt.show()


if WIDGETS_DISPONIBLES:
    # Cada slider controla un parametro de la vista, rho elige la trayectoria
    interact(
        vista_lorenz_interactiva,
        xmin =widgets.FloatSlider(value=-25, min=-80, max=20,  step=1),
        xmax =widgets.FloatSlider(value=25,  min=-20, max=80,  step=1),
        ymin =widgets.FloatSlider(value=-35, min=-100,max=20,  step=1),
        ymax =widgets.FloatSlider(value=35,  min=-20, max=100, step=1),
        zmin =widgets.FloatSlider(value=0,   min=0,   max=100, step=1),
        zmax =widgets.FloatSlider(value=60,  min=10,  max=180, step=1),
        elev =widgets.IntSlider(value=24,    min=0,   max=90,  step=2),
        azim =widgets.IntSlider(value=-56,   min=-180,max=180, step=4),
        salto=widgets.IntSlider(value=4,     min=1,   max=30,  step=1),
        rho  =widgets.Dropdown(options=[14.0, 28.0, 99.6], value=28.0)
    )
else:
    # Si ipywidgets no está instalado mostramos una vista fija equivalente
    print("ipywidgets no disponible. Mostrando vista fija equivalente.")
    vista_lorenz_interactiva()

### 4.6 Animación del avance de las iteraciones

**¿Por qué hemos implementado esta mejora?**  
El enunciado sugiere crear animaciones que ilustren el avance de las iteraciones. Una imagen final del atractor muestra el resultado acumulado, pero oculta la información dinámica: no se aprecia que la trayectoria tarda un tiempo en acercarse al atractor, ni que los lóbulos se van alternando de forma irregular. La animación permite observar dos aspectos que conectan con la teoría: primero, que existe una fase transitoria antes de que la trayectoria quede atrapada en el atractor; y segundo, que una vez dentro, la alternancia entre lóbulos es la manifestación visual del comportamiento caótico descrito en el Teorema de Picard-Lindelöf como dependencia sensible a las condiciones iniciales.

**¿Cómo lo hemos hecho?**  
Generamos la animación con `matplotlib.animation.FuncAnimation`, añadiendo un punto nuevo en cada fotograma. La cámara rota lentamente variando el ángulo azimutal en cada frame para revelar progresivamente la estructura tridimensional del atractor. El resultado se renderiza como HTML interactivo con `IPython.display.HTML`.

In [ ]:
def crear_animacion_lorenz(U_anim, t_anim=None, salto=15, intervalo_ms=30,
                           cola=300):
    """
    Animación 3D del atractor de Lorenz con mejoras visuales:
    - Intervalo completo t in [0, 100] en lugar de [0, 60]
    - Cola coloreada: los últimos `cola` puntos se muestran con gradiente azul
    - Punto cabeza en rojo marcando la posición actual
    - Contador de tiempo en el título
    - Rotación lenta de cámara para revelar la estructura 3D

    Parámetros
    ----------
    U_anim      : array (N, 3) — trayectoria completa
    t_anim      : array (N,)  — tiempos correspondientes (opcional, para el contador)
    salto       : submuestreo; salto=15 sobre h=0.01 → cada 0.15 unidades de tiempo
    intervalo_ms: milisegundos entre fotogramas
    cola        : número de puntos que forman la estela visible
    """
    V = U_anim[::salto]
    T = t_anim[::salto] if t_anim is not None else np.linspace(0, 100, len(V))
    n_frames = len(V)

    fig = plt.figure(figsize=(9, 7))
    ax  = fig.add_subplot(111, projection="3d")
    ax.set_xlim(U_anim[:, 0].min() - 1, U_anim[:, 0].max() + 1)
    ax.set_ylim(U_anim[:, 1].min() - 1, U_anim[:, 1].max() + 1)
    ax.set_zlim(U_anim[:, 2].min() - 1, U_anim[:, 2].max() + 1)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.view_init(elev=24, azim=-56)

    # Trayectoria de fondo muy tenue para dar contexto geométrico del atractor
    ax.plot(V[:, 0], V[:, 1], V[:, 2], lw=0.3, color=GRIS, alpha=0.18)

    # Elementos dinámicos: estela coloreada + punto cabeza
    estela, = ax.plot([], [], [], lw=1.1, color=AZUL, alpha=0.85)
    cabeza, = ax.plot([], [], [], "o", color=ROJO, markersize=5, zorder=10)
    titulo_txt = ax.set_title("", pad=8)

    def init():
        estela.set_data([], []); estela.set_3d_properties([])
        cabeza.set_data([], []); cabeza.set_3d_properties([])
        return estela, cabeza

    def actualizar(i):
        # Estela: últimos `cola` puntos hasta el frame actual
        inicio = max(0, i - cola)
        seg = V[inicio : i + 1]
        estela.set_data(seg[:, 0], seg[:, 1])
        estela.set_3d_properties(seg[:, 2])
        # Cabeza: posición actual
        cabeza.set_data([V[i, 0]], [V[i, 1]])
        cabeza.set_3d_properties([V[i, 2]])
        # Título con contador de tiempo
        titulo_txt.set_text(f"Atractor de Lorenz — t = {T[i]:.1f}")
        # Rotación suave de cámara (360° completas a lo largo de la animación)
        ax.view_init(elev=24, azim=-56 + 360 * i / n_frames)
        return estela, cabeza

    anim = animation.FuncAnimation(
        fig, actualizar, init_func=init,
        frames=n_frames, interval=intervalo_ms, blit=False
    )
    plt.close(fig)
    return anim


# Animamos la trayectoria COMPLETA t in [0, 100]
# salto=15 sobre 10 001 puntos → ~667 fotogramas, duración ~20 s a 30 ms/frame
animacion = crear_animacion_lorenz(U, t_anim=t, salto=15, intervalo_ms=30, cola=250)
HTML(animacion.to_jshtml())


### 4.7 Mapa local de sensibilidad a perturbaciones

**¿Por qué hemos implementado esta mejora?**  
El Apartado 2 ilustra el efecto mariposa comparando dos trayectorias concretas. Este aporte lo generaliza: en lugar de dos puntos, estudiamos una vecindad completa de la condición inicial base. Para cada perturbación $(\Delta x_0, \Delta y_0)$ en una cuadrícula de $41\times 41$ puntos alrededor de $(3,15,1)$ medimos cuánto se separa la trayectoria de la original al cabo de $t=30$. El resultado conecta con otras prácticas de la asignatura: de la misma forma que coloreamos el plano según la raíz de convergencia en el método de Newton-Raphson, aquí coloreamos según la separación producida. La escala logarítmica permite apreciar diferencias de varios órdenes de magnitud en una misma figura.

**¿Cómo lo hemos hecho?**  
Las $41^2=1681$ integraciones son completamente independientes entre sí, por lo que las paralelizamos con `joblib` repartiendo el trabajo entre los núcleos disponibles. Cada integración reutiliza nuestra función `resolver_rk4` con $h=0.02$ y el resultado se organiza en una matriz que se visualiza con `imshow` de Matplotlib usando el colormap `magma`.

**Interpretación.**  
El eje horizontal representa perturbaciones en $x_0$ y el vertical en $y_0$, ambos en el rango $\pm 2\times 10^{-3}$ alrededor de la condición base. El punto central (marcado en cian) es la propia condición base, con separación cero. Los colores oscuros indican separaciones pequeñas al cabo de $t=30$ y los claros separaciones grandes. Lo más llamativo es que el mapa no es suave: no hay una transición gradual del centro hacia los bordes, sino una estructura irregular con zonas oscuras y claras intercaladas sin patrón aparente. Esto es el caos: condiciones iniciales que están prácticamente en el mismo punto pueden dar separaciones muy distintas dependiendo de la dirección exacta de la perturbación. El mapa muestra que la sensibilidad no es isótropa, hay direcciones en las que el sistema diverge mucho más rápido que en otras.

In [ ]:
def _simular_sensibilidad(dx, dy, u0_base, tiempo, h_local, params):
    """Función auxiliar para joblib: integra desde u0_base + (dx, dy, 0)."""
    u0 = u0_base + np.array([dx, dy, 0.0])   # solo variamos x e y
    _, U_tmp = resolver_rk4(lorenz, u0, intervalo=(0.0, tiempo),
                             h=h_local, **params)
    return U_tmp[-1]


def mapa_sensibilidad_local(delta=2e-3, n=41, tiempo=30.0):
    """Mapa n×n de log10 de la separación final respecto a la trayectoria base."""
    h_local = 0.02
    _, U_central = resolver_rk4(lorenz, u0_base, intervalo=(0.0, tiempo),
                                 h=h_local, **params_caoticos)
    final_central = U_central[-1]

    dxs = np.linspace(-delta, delta, n)
    dys = np.linspace(-delta, delta, n)
    combinaciones = [(dx, dy) for dy in dys for dx in dxs]

    # Lanzamos las n² integraciones en paralelo para reducir el tiempo de espera
    finales = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_simular_sensibilidad)(dx, dy, u0_base, tiempo, h_local, params_caoticos)
        for dx, dy in combinaciones
    )

    # Log10 para que los órdenes de magnitud se vean más claramente en el mapa
    M = np.array([
        np.log10(np.linalg.norm(f - final_central) + 1e-14) for f in finales
    ]).reshape(n, n)

    return dxs, dys, M


print("Calculando mapa de sensibilidad (paralelizado)...")
dxs, dys, M = mapa_sensibilidad_local(delta=2e-3, n=41, tiempo=30.0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    M, origin="lower",
    extent=[dxs[0]*1e3, dxs[-1]*1e3, dys[0]*1e3, dys[-1]*1e3],
    cmap="magma", aspect="auto"
)
ax.scatter([0], [0], color="cyan", edgecolor="black", s=70,
           label="condición base", zorder=5)
ax.set_title(r"Mapa de sensibilidad local ($t=30$)")
ax.set_xlabel(r"perturbación en $x_0$  ($\times 10^{-3}$)")
ax.set_ylabel(r"perturbación en $y_0$  ($\times 10^{-3}$)")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r"$\log_{10}\|\Delta u(t=30)\|_2$")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Rango del mapa: [{M.min():.2f}, {M.max():.2f}] en escala log₁₀")

---
## Conclusiones

En esta práctica hemos resuelto el sistema de Lorenz usando el método de Runge-Kutta clásico con longitud de paso $h=0.01$. Para los parámetros con los que Lorenz identificó el comportamiento caótico en 1963,

$$
\sigma=10,\qquad \rho=28,\qquad \beta=\frac{8}{3},
$$

la solución aproximada dibuja el atractor extraño de Lorenz, cuya dimensión fractal se estima en torno a $d_F\approx 2.06$. Esto nos ha permitido comprobar visualmente que una solución de un PVI puede permanecer acotada sin converger a un punto fijo ni seguir una órbita periódica simple.

Al repetir el proceso con distintas condiciones iniciales, hemos observado que pequeñas perturbaciones producen recorridos completamente distintos a largo plazo. La estimación del mayor exponente de Lyapunov, $\lambda_1\approx 0.905$ nat/s, coherente con el valor de referencia $0.906$, cuantifica esta divergencia e implica un horizonte de predictibilidad finito aunque el sistema sea determinista. Esta parte nos ha ayudado a entender que el llamado efecto mariposa no es un fallo del método numérico, sino una propiedad de la propia dinámica del sistema.

La comparación de parámetros también muestra que $\rho$ determina el tipo de comportamiento dinámico. Para $\rho=14$ la trayectoria tiende a estabilizarse, mientras que para $\rho=28$ aparece el atractor clásico y para $\rho=99.6$ la dinámica ocupa una región mucho más amplia y compleja. El diagrama de bifurcaciones y el umbral $\rho_H\approx 24.74$ permiten interpretar estas transiciones de forma sistemática, mostrando que los parámetros no solo modifican la escala de las soluciones, sino también su comportamiento cualitativo.

Desde el punto de vista del temario de PVI, esta práctica nos ha servido para entender que resolver un problema de valores iniciales no consiste únicamente en aplicar una fórmula y obtener una gráfica. Primero hemos tenido que plantear el sistema en la forma general

$$
\mathbf{u}'(t)=\mathbf{F}(t,\mathbf{u}(t)), \qquad \mathbf{u}(0)=\mathbf{u}_0,
$$

identificar el campo vectorial, justificar mediante el Teorema de Picard-Lindelöf que el problema está bien planteado y después elegir un método numérico adecuado para aproximar la solución. Esto nos ha ayudado a ver que los resultados teóricos de existencia y unicidad no son meras condiciones formales, sino la base que permite empezar a estudiar el problema con sentido.

También hemos comprendido mejor el papel de los métodos de un paso, por ejemplo viendo como RK4 se muestra como una forma de combinar cuatro pendientes para aproximar la evolución de la solución con mayor precisión que métodos más simples como Euler. La verificación del orden de convergencia confirma el orden 4 predicho por la teoría, siempre que la comparación se haga antes de que la sensibilidad caótica amplifique las diferencias numéricas. Por tanto, la práctica nos ha obligado a distinguir entre error de discretización, tamaño del paso, estabilidad del método y comportamiento propio del modelo.

En definitiva, esta práctica nos ha servido para conectar la teoría de los PVI con un ejemplo donde los fenómenos numéricos se ven de forma muy clara. Hemos aprendido que un método puede ser convergente y estar correctamente implementado, pero aun así no ofrecer predicciones fiables a largo plazo si el sistema es caótico. Por eso, más que obtener solo una solución aproximada, hemos aprendido a interpretar hasta dónde podemos confiar en ella y qué información cualitativa podemos extraer del modelo.
"""



---
## Referencias

* **Piñar, M. A. (2026).** *Métodos Numéricos II. Tema 3: Métodos Numéricos para resolver PVI*. Departamento de Matemática Aplicada, Facultad de Ciencias, Universidad de Granada. Material de clase.

* **Lorenz, E. N. (1963).** Deterministic Nonperiodic Flow. *Journal of the Atmospheric Sciences*, 20(2), 130-141.


* **Mene Hevia, A. (2019).** Cálculo de los exponentes característicos de Lyapunov. Aplicación a la detección de caos. Trabajo de Fin de Grao, Universidade de Santiago de Compostela, Grao de Matemáticas. https://minerva.usc.gal/rest/api/core/bitstreams/0887455f-1bf3-4405-8def-95040d5b01e0/content

* **De Oro García, J. (2024).** Los límites de la predicción meteorológica: las ecuaciones de Lorenz y el efecto mariposa. Trabajo de Fin de Grado, Universidad Complutense de Madrid, Facultad de Ciencias Físicas. https://docta.ucm.es/rest/api/core/bitstreams/869b643f-a92c-4aad-8ffb-9d63a45388fe/content

* **Chaos in the Lorenz System Using 4th-Order Runge-Kutta Method to Analyze Parameter- and Initial Condition-Dependent Dynamics.** *ResearchGate*. https://www.researchgate.net/publication/396992383_Chaos_in_the_Lorenz_System_Using_4th-Order_Runge-Kutta_Method_to_Analyze_Parameter-_and_Initial_Condition-Dependent_Dynamics

* **Wikipedia contributors. (2025).** Lorenz system. *Wikipedia*. https://en.wikipedia.org/wiki/Lorenz_system [Referencia proporcionada por el enunciado]

* **Weisstein, E. W.** Lorenz Attractor. *MathWorld*. https://mathworld.wolfram.com/LorenzAttractor.html [Referencia proporcionada por el enunciado]


* **Project Jupyter.** ipywidgets documentation. https://ipywidgets.readthedocs.io/

* **Joblib contributors.** joblib: running Python functions as pipeline jobs. https://joblib.readthedocs.io/

* **Anthropic. (2025).** *Claude* (Claude Sonnet 4.6) [LLM]. Utilizado como apoyo para revision de codigo, sugerencias de mejora y contraste de explicaciones teoricas. El codigo, las visualizaciones y los analisis han sido escritos y verificados por las autoras. https://claude.ai